# [RQ_2 · BỔ SUNG] Config C — Tuning và so sánh 4 mô hình

Mục tiêu: tinh chỉnh và so sánh Linear Regression, Random Forest, Gradient Boosting và MLP trên Config C.

Config C gồm đặc trưng thời gian, ngày lễ, thời tiết và lịch sử giao thông. Tuning dùng các fold theo thời gian trong train. Validation dùng để so sánh mô hình sau tuning. Giữ test riêng cho đánh giá cuối.

Ghi nhật ký từng ứng viên và fold trong lúc tuning.


**Nguồn dữ liệu chung (19/11/2026):** Code đọc dữ liệu đã chuyển sang `src.experiment_data.load_experiment_splits`. `processed` dùng snapshot bàn giao; `raw` dùng cho CV trước điền thiếu; `filled` giữ schema nhãn và học điền từ train. Các output/metrics đang hiển thị là kết quả lịch sử, chưa chạy lại sau thay đổi này.


In [ ]:
CONFIG_NAME = "C"
TARGET = "traffic_volume"
PRIMARY_METRIC = "MAE"
RANDOM_STATE = 42

MODEL_NAMES = [
    "Linear Regression",
    "Random Forest",
    "Gradient Boosting",
    "MLP",
]

DATA_PARAMS = {
    "mask_stuck_temp": False,
    "complete_holidays": False,
}


In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone, timedelta


def find_project_root():
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        if (
            (folder / "src" / "features.py").is_file()
            and (folder / "src" / "tuning.py").is_file()
        ):
            return folder

    raise FileNotFoundError("Không tìm thấy thư mục gốc của dự án.")


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src.data import DATA_PATH, fill_missing_weather

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "13-11-2026_ConfigC_4Model_Vu"
)

RUN_ID = datetime.now(
    timezone(timedelta(hours=7))
).strftime("%Y%m%d_%H%M%S")

RUN_DIR = EXPERIMENT_DIR / "outputs" / RUN_ID
LOG_PATH = (
    EXPERIMENT_DIR
    / "logs"
    / "tuning_progress"
    / f"{RUN_ID}.csv"
)
FIGURE_DIR = EXPERIMENT_DIR / "figures" / RUN_ID

print("Project:", PROJECT_ROOT)
print("Data:", DATA_PATH)
print("Run:", RUN_ID)


## Đọc dữ liệu và tạo đúng Config C

Tạo đặc trưng từ ba phần dữ liệu theo đúng thứ tự thời gian, sau đó chỉ giữ các giờ đã có đủ ba đặc trưng lịch sử.


In [ ]:
from src.experiment_data import load_experiment_splits
raw_splits = load_experiment_splits(
    path=DATA_PATH,
    **DATA_PARAMS, stage="raw")

filled_splits = fill_missing_weather(raw_splits)
featured_splits = features.build_features(filled_splits)

FEATURE_COLUMNS = list(features.CONFIG_C)

train_all = featured_splits["train"]
validation_all = featured_splits["validation"]

train = train_all.loc[
    features.has_history(train_all)
].copy()

validation = validation_all.loc[
    features.has_history(validation_all)
].copy()

assert not train.empty
assert not validation.empty
assert train["date_time"].is_monotonic_increasing
assert validation["date_time"].is_monotonic_increasing
assert train["date_time"].max() < validation["date_time"].min()

for name, frame in {
    "train": train,
    "validation": validation,
}.items():
    missing = set(FEATURE_COLUMNS + [TARGET]) - set(frame.columns)
    assert not missing, f"{name} thiếu cột: {sorted(missing)}"

print("Số đặc trưng Config C:", len(FEATURE_COLUMNS))
print("Train:", len(train), train["date_time"].min(), "→", train["date_time"].max())
print(
    "Validation:",
    len(validation),
    validation["date_time"].min(),
    "→",
    validation["date_time"].max(),
)


## Tuning 4 mô hình

Thử siêu tham số bằng các fold theo thời gian bên trong train. Nhật ký được nối thêm sau mỗi fold và mỗi ứng viên.


In [ ]:
from src.tuning import run_config
from src.trial_logger import TrialProgressLogger

trial_logger = TrialProgressLogger(
    path=LOG_PATH,
    run_id=RUN_ID,
)

try:
    results = run_config(
        config=CONFIG_NAME,
        features=FEATURE_COLUMNS,
        train=train,
        target=validation,
        model_names=MODEL_NAMES,
        raw_train=raw_splits["train"],
        progress_callback=trial_logger,
    )
finally:
    trial_logger.close()


## Kết quả validation và các ứng viên

Xếp hạng các mô hình theo MAE validation. Bảng ứng viên cho biết MAE từng fold và MAE CV trung bình.


In [ ]:
from IPython.display import display
from src.models import results_table

summary = (
    results_table(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(summary.round(4))

for item in results:
    print(f"\n{item['model']} — các ứng viên đã thử")
    display(item["search"].scores)


## Lưu kết quả và biểu đồ

Lưu metrics, siêu tham số tốt nhất, dự đoán validation và biểu đồ 168 giờ đầu.


In [ ]:
from src.models import save_results
from src.experiment_plots import save_validation_plot

save_results(
    version="ConfigC",
    results=results,
    target=validation,
    output_dir=RUN_DIR,
)

figure_path = FIGURE_DIR / "config_c_4_models_168h.png"
save_validation_plot(
    results=results,
    target=validation,
    output_path=figure_path,
    hours=168,
)

print("Kết quả:", RUN_DIR)
print("Biểu đồ:", figure_path)


## Nhật ký sau khi hoàn tất

Ghi một bản tóm tắt cho từng mô hình sau khi tuning và đánh giá validation kết thúc. Nhật ký tiến trình trong lúc tuning được lưu riêng ở CSV.


In [ ]:
import json
from src.experiment_logger import save_experiment_log

run_date = f"{RUN_ID[:4]}-{RUN_ID[4:6]}-{RUN_ID[6:8]}"
completed_log_dir = (
    EXPERIMENT_DIR
    / "logs"
    / "completed"
    / RUN_ID
)

for item in results:
    best_params = json.dumps(
        item["search"].best_params_,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )

    save_experiment_log(
        model=f"Config C — {item['model']}",
        features=FEATURE_COLUMNS,
        results=item["metrics"],
        observations=[
            f"MAE trung bình CV: {-item['search'].best_score_:.4f}",
            f"Params tốt nhất: {best_params}",
            f"Số mẫu train: {len(train)}",
            f"Số mẫu validation: {len(validation)}",
        ],
        problems=[],
        next_steps=[
            "Giữ test riêng cho đánh giá cuối sau khi chọn mô hình."
        ],
        data_split={
            "Train": 70,
            "Validation": 15,
            "Test": 15,
        },
        split_strategy="Chronological",
        experiment_date=run_date,
        output_dir=completed_log_dir,
    )


<!-- proposal-assessment -->
## Đánh giá đối với proposal — RQ_2

**Vai trò: BỔ SUNG.** Mã đối chiếu: [E07](../../docs/experiment_tracking/EXPERIMENTS_BY_DATE_AND_RQ.md). Đây là hướng mở rộng/kiểm tra bổ sung, không thay thế đối chứng chính A→B của RQ_2.

- **Câu hỏi:** Trong cùng cấu hình có lịch sử C30, pipeline nào tốt nhất sau tuning?
- **Đánh giá:** Phục vụ phần mở rộng có lịch sử và so mô hình trong cùng C30. RF đứng đầu bảng lưu với MAE 195,336/RMSE 321,983 trên 6.050 giờ validation.
- **Giới hạn kết luận:** Không phải C74 của Hào. Bảng C riêng chưa cô lập giá trị lịch sử nếu thiếu B cùng cohort; không dùng để xếp hạng thuật toán giữa các pipeline khác feature/preprocessing.
- **Bằng chứng:** [E07: ConfigC_validation.csv](outputs/20261113_220619/metrics/ConfigC_validation.csv).

Đánh giá trên dựa vào hiện vật lịch sử đã đối chứng, không phải kết quả chạy mới. Code lấy dữ liệu đã được chuẩn hóa; chưa huấn luyện lại hoặc chấm test.
